# 📊 04 — Model Evaluation

Evaluate all trained models on the test set:
- Accuracy, Precision, Recall, F1-Score
- Confusion Matrix
- ROC-AUC Curve
- Precision-Recall Curve
- Model Comparison

In [ ]:
import sys
sys.path.append('..')

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import matplotlib.pyplot as plt

import config
from src.models.model_factory import ModelFactory
from src.data.dataset import create_generators
from src.evaluation.metrics import evaluate_model
from src.evaluation.visualizations import (
    plot_confusion_matrix,
    plot_roc_curve,
    plot_precision_recall_curve,
    plot_model_comparison,
)

%matplotlib inline
plt.style.use('dark_background')
print('✓ Libraries loaded')

## 1. Load Trained Models

In [ ]:
models = {}
for name in ModelFactory.list_models():
    try:
        model = ModelFactory.create(name)
        model.load()  # Load from default path
        models[name] = model
        print(f'✓ Loaded: {name}')
    except Exception as e:
        print(f'✗ Could not load {name}: {e}')

print(f'\nLoaded {len(models)} model(s)')

## 2. Evaluate Each Model

In [ ]:
all_results = {}

for name, model in models.items():
    _, _, test_gen = create_generators(model_name=name)
    result = evaluate_model(model, test_gen, verbose=True)
    all_results[name] = result

## 3. Confusion Matrices

In [ ]:
for name, result in all_results.items():
    cm = result['metrics']['confusion_matrix']
    path = plot_confusion_matrix(cm, model_name=name)
    print(f'Saved: {path}')
    
    img = plt.imread(path)
    plt.figure(figsize=(7, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.show()

## 4. ROC Curves

In [ ]:
for name, result in all_results.items():
    roc = result['metrics']['roc_curve']
    auc = result['metrics']['auc_roc']
    path = plot_roc_curve(roc['fpr'], roc['tpr'], auc, model_name=name)
    print(f'Saved: {path}')
    
    img = plt.imread(path)
    plt.figure(figsize=(7, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.show()

## 5. Precision-Recall Curves

In [ ]:
for name, result in all_results.items():
    pr = result['metrics']['pr_curve']
    path = plot_precision_recall_curve(pr['precision'], pr['recall'], model_name=name)
    print(f'Saved: {path}')
    
    img = plt.imread(path)
    plt.figure(figsize=(7, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.show()

## 6. Model Comparison

In [ ]:
comparison = {}
for name, result in all_results.items():
    m = result['metrics']
    comparison[name] = {
        'accuracy': m['accuracy'],
        'precision': m['precision'],
        'recall': m['recall'],
        'f1_score': m['f1_score'],
        'auc_roc': m['auc_roc'],
    }

path = plot_model_comparison(comparison)
print(f'Saved: {path}')

img = plt.imread(path)
plt.figure(figsize=(12, 6))
plt.imshow(img)
plt.axis('off')
plt.show()

## 7. Summary Table

In [ ]:
import pandas as pd

df = pd.DataFrame(comparison).T
df.index.name = 'Model'
df = df.round(4)
print('\n' + '='*60)
print('  MODEL COMPARISON SUMMARY')
print('='*60)
print(df.to_string())
print(f'\n  Best model by accuracy: {df["accuracy"].idxmax().upper()}')
print(f'  Best model by F1-score: {df["f1_score"].idxmax().upper()}')
print(f'  Best model by AUC-ROC:  {df["auc_roc"].idxmax().upper()}')

---
**Next:** Proceed to `05_gradcam_visualization.ipynb` for explainability analysis.